# Writing Web Crawlers

This small script that extract all the URL links in a Wikipedia page and print them

In [ ]:
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup

URL = 'http://en.wikipedia.org/wiki/Kevin_Bacon'

# Mimic a real browser request
req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})

try:
    html = urlopen(req)
except (HTTPError, URLError) as e:
    print(f'Failed to fetch page: {e}')
    exit()

bs = BeautifulSoup(html.read(), 'html.parser')

for link in bs.find_all('a'):
    if 'href' in link.attrs:
        print(link.attrs['href'])



This store all the links in the Wikipedia page that contains the word '**(film)**' store them in a txt file

In [ ]:
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup

URL = 'http://en.wikipedia.org/wiki/Kevin_Bacon'

# Mimic a real browser request
req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})

try:
    html = urlopen(req)
except (HTTPError, URLError) as e:
    print(f'Failed to fetch page: {e}')
    exit()

bs = BeautifulSoup(html.read(), 'html.parser')

with open('testlinks.txt','w') as file:
    for link in bs.find_all('a'):
        if 'href' in link.attrs and '(film)' in link.attrs['href']:
            file.write(link.attrs['href']+"\n")

So this script bellow extract all the article links in a page this is how it works
There is a patterns between “article links” and “other links" 
that they all have three things in common :

• They all in a **div** tag with the **id** is **bodyContent**. so all the link article are stored in **body** inside `<div id = bodyContent>` tag
• The **URLs** do **not** **contain colons**.
• The **URLs** **begin** with **/wiki/**.

So this how we will **filter** **first** we will find the `<div>` tag that has the `id = bodyContent` like this 

```python
bs.find('div', {'id':'bodyContent'})
```
and then inside the `<div id = bodyContent>` tag will will **find all** the **links href** that contain the **keyword**  : **`wiki`** 

with regular expression :  this `r"^/wiki/[^:]*$"` or this `^(/wiki/)((?!:).)*$`

NOTE : 


```python
soup.find("a") # --> Returns ONLY the first match, A single Tag object Or `None` if nothing found

soup.find_all("a") # --> Returns ALL matching elements in A list (ResultSet) 
```

In [ ]:
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup
import re

URL = 'http://en.wikipedia.org/wiki/Kevin_Bacon'

# Mimic a real browser request
req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})

try:
    html = urlopen(req)
except (HTTPError, URLError) as e:
    print(f'Failed to fetch page: {e}')
    exit()

bs = BeautifulSoup(html.read(), 'html.parser')

for link in bs.find('div', {'id':'bodyContent'}).find_all('a', href=re.compile(r"^/wiki/[^:#]*$")):
    print(link.attrs['href'])
    # print(link) --> This will show all the tag like this <a href="/wiki/Philadelphia" title="Philadelphia">Philadelphia</a>



In [ ]:

articleUrl = '/wiki/Kevin_Bacon'

html = 'http://en.wikipedia.org{}'.format(articleUrl) # The {} is a placeholder that gets replaced by whatever is inside .format(....).

print(html)

html = 'http://en.wikipedia.org'+articleUrl

print(html)

html = f'http://en.wikipedia.org{articleUrl}'
print(html)

So now let's create a new script more advanced then before so now this script will contain a function called  **getLinks** that takes a **Wikipedia article URL** and **extract & returns** the list of URLs inside that **URL page** keep  in mind all the URL articles are in the same format `/wiki/<article_name>`

and with that function  another function that calls  `getLinks` and give it a **URL link** and then chooses a random article link from the returned list, and calls **getLinks** again, until it extract 10 random links
this is what it does :

It starts on Kevin Bacon's Wikipedia page, picks a **random link** on that page, goes to that page, picks another random link, and keeps going forever — like a random walk through Wikipedia.

you took a random walk through a website, going from link to link

In [ ]:
import re
import random
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup


def getlinks(articleUrl):
    URL = f'http://en.wikipedia.org{articleUrl}'
    
    req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        html = urlopen(req)
    except (HTTPError, URLError) as e:
        print(f'Failed to fetch page: {e}')
        exit()

    bs = BeautifulSoup(html.read(), 'html.parser')

    return [link for link in bs.find('div', {'id':'bodyContent'}).find_all('a', href=re.compile(r"^/wiki/[^:#]*$"))]
        
    

articleUrl = '/wiki/Kevin_Bacon'
links_list = getlinks(articleUrl)
for i in range(10):
    link = links_list[random.randint(0, len(links_list)-1)].attrs['href'] # takes a random link from the link lists NOTE: extract the value of href attributes
    print(link)
    links_list = getlinks(link) # extract all the links  inside the random link that we extract it



To avoid crawling the same page twice, we will store all the links that are we discover in a **set**. 

so we take a **link** we see if it's in the **set** if not we store it else we move to another 

```py
Find all <a> tags where href starts with /wiki/
        ↓
Check if href attribute actually exists
        ↓
Check if we haven't visited it before
        ↓
Print it, add to pages set, crawl it recursively



The Flow

Homepage
    ↓ finds /wiki/Python
        ↓ finds /wiki/Programming
            ↓ finds /wiki/Computer
                ↓ ... goes deeper and deeper
```

In [ ]:
import re
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup

pages = set()

def getLinks(pageUrl):

    URL = f'http://en.wikipedia.org{pageUrl}'
    req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    html = urlopen(req) 
    bs = BeautifulSoup(html, 'html.parser')

    for link in bs.find_all('a', href=re.compile('^(/wiki/)')): # gets the tag <a> : links in a wiki page
        newPage = link.attrs['href']  # get the value of href attributes
        if newPage not in pages: # see if the value is in the set()
            print(newPage) 
            pages.add(newPage)
            getLinks(newPage) # after store it in a set() we call the funtion again and repeat all the steps 'recursion'

getLinks('')

Collecting Data Across an Entire Site

Web crawler don't have only one functionality which is hop from one page to another but what if you be able to do something on the page while you're there before going to another

So let's build a scraper that collect the **title**, the **first paragraph** **of content** and the **link to edit the page (if available)**.

In [ ]:
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup
import re

pages = set()

def links_extract(url):
    #URL = 'http://en.wikipedia.org/wiki/Kevin_Bacon'
    #URL = 'https://en.wikipedia.org/wiki/Personal_computer'
    #URL = 'https://en.wikipedia.org/wiki/Python_(programming_language)'
    #URL = 'https://en.wikipedia.org/wiki/Function_(computer_programming)'
    #URL = 'https://en.wikipedia.org/wiki/File:Orbit_of_274301_Wikipedia.svg'

    URL = f'http://en.wikipedia.org{url}'

    # Mimic a real browser request
    req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})

    try:
        html = urlopen(req)
    except (HTTPError, URLError) as e:
        print(f'Failed to fetch page: {e}')
        exit()

    bs = BeautifulSoup(html.read(), 'html.parser')
    try :
        title = bs.find('h1')
        print(f'The Title of the Page is : {title.span.get_text()}')


        inner_div = bs.find('div', {'id': 'mw-content-text'}).find('div', {'class': 'mw-content-ltr mw-parser-output'})
        paragraphs = inner_div.find_all('p')
        frist_paragraph = paragraphs[1]  # index 1 = second element
        print(f'The first paragraph is : {frist_paragraph.text}')

        edit = bs.find(id = 'ca-edit').find('a')
        print(f'The link to edit this page is : {edit.attrs['href']}')
    except AttributeError:
        print('='*60+'\n')
        print(f'Have an Error on this URL : {URL} so move on')
    
    # now find the new URLs, extract them and then store them
    for link in bs.find_all('a', href=re.compile('^(/wiki/)')): # gets the tag <a> : links in a wiki page
        newPage = link.attrs['href']  # get the value of href attributes
        if newPage not in pages: # see if the value is in the set()
            print('='*60+'\n')
            print(f'The Url selected is : {newPage}') 
            pages.add(newPage)
            links_extract(newPage) # after store it in a set() we call the funtion again and repeat all the steps 'recursion'

links_extract('/wiki/Personal_computer')

Crawling Across the Internet

In [ ]:
from urllib.parse import urlparse

url = "https://www.example.com:8080/path?q=1"

netloc = urlparse(url).netloc  # "www.example.com:8080"
scheme = urlparse(url).scheme  # "https"

print(netloc)
print(scheme)

So we will create a function called `get_internal_links` Takes as arguments : 
a **BeautifulSoup** object and the **URL page**
And this function will Retrieves all the Internal links found on a given **URL** and store them in a **set** **list**  and then return 

so it will extract all the **href values** inside the `<a>` tag and then see if the **URL** is internal it will store it or if the URL is external it will pass it  

In [ ]:
from urllib.parse import urlparse
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup


def get_internal_links(bs,URL):
    netloc = urlparse(URL).netloc #  "www.example.com"
    scheme = urlparse(URL).scheme #  "https"

    internal_links = set()

    for link in bs.find_all('a'): # find every <a> tag in the HTML
        if not link.attrs.get('href'): # skip <a> tags with no href
            continue
        parsed = urlparse(link.attrs['href'])  # parse the href value
        
        if parsed.netloc == '':
            fix_url = f'{scheme}://{netloc}/{link.attrs["href"].strip("/")}'
            internal_links.add(fix_url)
        
        elif parsed.netloc == netloc:
            internal_links.add(link.attrs['href'])
    
    return list(internal_links)


URL = 'https://en.wikipedia.org/'

# Mimic a real browser request
req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})

try:
    html = urlopen(req)
except (HTTPError, URLError) as e:
    print(f'Failed to fetch page: {e}')
    exit()

bs = BeautifulSoup(html.read(), 'html.parser')

print(get_internal_links(bs,URL))

now we will create a function called `get_external_links` Takes as arguments : 
a **BeautifulSoup** object and the **URL page**
And this function will Retrieves all the external links found on a given **URL** and store them in a **set** **list**  and then return 

so it will extract all the **href values** inside the `<a>` tag and then see if the **URL** is external it will store it or if the URL is internal it will pass it  

In [ ]:
from urllib.parse import urlparse
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup


def get_external_links(bs,URL):
    netloc = urlparse(URL).netloc #  "www.example.com"

    external_links = set()

    for link in bs.find_all('a'): # find every <a> tag in the HTML
        if not link.attrs.get('href'): # skip <a> tags with no href
            continue
        
        parsed = urlparse(link.attrs['href'])  # parse the href value
        
        if parsed.netloc != netloc and parsed.netloc != '': 
            external_links.add(link.attrs['href'])
    
    return list(external_links)


URL = 'https://en.wikipedia.org/'

# Mimic a real browser request
req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})

try:
    html = urlopen(req)
except (HTTPError, URLError) as e:
    print(f'Failed to fetch page: {e}')
    exit()

bs = BeautifulSoup(html.read(), 'html.parser')

print(get_external_links(bs,URL))

Now we can use the `get_external_links` function to crawler all over the web so we can make a **recursive function** that calls it the ``get_external_links`` and calls it self non stop untill we stop it   

This script is a random web crawler that starts from a given URL and endlessly jumps from website to website by following random external links

In [ ]:
from urllib.parse import urlparse
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup
import random

def get_external_links(bs,URL):
    netloc = urlparse(URL).netloc #  "www.example.com"

    external_links = set()

    for link in bs.find_all('a'): # find every <a> tag in the HTML
        if not link.attrs.get('href'): # skip <a> tags with no href
            continue
        
        parsed = urlparse(link.attrs['href'])  # parse the href value
        
        if parsed.netloc != netloc and parsed.netloc != '': 
            external_links.add(link.attrs['href'])
    
    return list(external_links)



def get_random_link(URL):
    req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        html = urlopen(req)
    except (HTTPError, URLError) as e:
        print(f'Failed to fetch page: {e}')
        exit()
    bs = BeautifulSoup(html.read(), 'html.parser')

    externalLinks = get_external_links(bs,URL)

    if len(externalLinks) == 0:
        return 0
    else:
        return random.choice(externalLinks)



def crawler_the_web(URL):
    externalLink = get_random_link(URL)
    print(f'start crawling is : {externalLink}')
    crawler_the_web(externalLink)

    


crawler_the_web('https://en.wikipedia.org')

Web Crawling Models

crawler and scraper multiple/diffrent sites 

create a 2 scrapers functions 

So now for example we will create 2 functions scrapers one that scrape **CNN site** articles and the other scrape **Brookings site** articles. The 2 functions extract the **URL**, **title** and the **body** of the article so we will create a **class** named **Content** that has those **fields**


In [ ]:
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup


class Content:
    def __init__(self,title,url,body):
        self.title = title
        self.body = body
        self.url = url
    
    def print(self):
        print(f'TITLE: {self.title}')
        print(f'URL: {self.url}')
        print(f'BODY: {self.body}')

def scrapeCNN(URL):
    req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        html = urlopen(req)
    except (HTTPError, URLError) as e:
        print(f'Failed to fetch page: {e}')
        exit()
    bs = BeautifulSoup(html.read(), 'html.parser')
    title = bs.find('h1').get_text()
    body = bs.find('div', {'class': 'article__content'})
    return Content(title, url, body)

def scrapeBrookings(URL):
    req = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        html = urlopen(req)
    except (HTTPError, URLError) as e:
        print(f'Failed to fetch page: {e}')
        exit()
    bs = BeautifulSoup(html.read(), 'html.parser')
    title = bs.find('h1')
    body = bs.find('div', {'class': 'post-body'})
    return Content(title, url, body)

url = 'https://www.brookings.edu/research/robotic-rulemaking/'
content = scrapeBrookings(url)
content.print()

url = 'https://www.cnn.com/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html'
content = scrapeCNN(url)
content.print()

TITLE: 
      Dogecoin jumps after Elon Musk replaces Twitter bird with Shiba Inu
    
URL: https://www.cnn.com/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html
BODY: <div class="article__content" data-editable="content" data-reorderable="content" itemprop="articleBody">
<div class="source vossi-source inline-placeholder" data-article-gutter="true" data-component-name="source" data-uri="cms.cnn.com/_components/source/instances/source-h_584e6a1a2f664c1f31e6c8935bf95f83@published">
<cite class="source__cite vossi-source__cite">
<span class="source__location vossi-source__location" data-editable="location">New York</span>
<span class="source__text" data-editable="source">CNN</span>
         — 
    </cite>
</div>
<p class="paragraph inline-placeholder vossi-paragraph" data-article-gutter="true" data-component-name="paragraph" data-editable="text" data-uri="cms.cnn.com/_components/paragraph/instances/paragraph-ba14eef1fe8a71b4968679889def58eb@published">
            Twitter’s trad



So now i will create simple and reusable web scraper built with Python and BeautifulSoup that can extract content from multiple websites using CSS selectors. 

The Problem with the Previous Approach
In the previous version, we wrote a separate function for every website — scrapeBrookings(), scrapeCNN(), and so on. The logic inside each function was essentially identical, the only difference being the CSS selectors used to locate the title and body on each site. This meant duplicated code that was hard to scale — adding a new website required writing a whole new function.

The New Approach
This version solves that by combining all scraping logic into one reusable function — getContent. Instead of hardcoding the selectors, the user simply defines a list of Website objects, each holding:

- The site's base URL

- The CSS selector for the title

- The CSS selector for the body

How it works

The project is built around 3 classes:

- **`Content`** — stores the scraped data (title, body, url) and prints it
- **`Website`** — a data container that holds a site's name, base url, and CSS selectors for where to find the title and body
- **`Crawler`** — handles the actual scraping logic:
  - `getPage` fetches and parses the HTML of a given URL
  - `safeGet` uses CSS selectors to safely extract text from the parsed HTML
  - `getContent` combines both to scrape a full page and return a `Content` object

Usage

Define your target websites with their CSS selectors in `siteData`, then call `Crawler.getContent` with the website and the path you want to scrape:

```python
Crawler.getContent(siteData[0], '/wiki/Python_(programming_language)').print()
```

---


In [8]:
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup


class Content:
    def __init__(self,title,body,url):
        self.title = title
        self.body = body
        self.url = url
    
    def print(self):
        print(f'TITLE: {self.title}')
        print(f'BODY: {self.body}')
        print(f'URL: {self.url}')

class Website:
    def __init__(self,name,url,titleTag,bodyTag):
        self.name = name
        self.url = url
        self.titleTag = titleTag
        self.bodyTag = bodyTag

class Crawler:
    def getPage(url):
        req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            html = urlopen(req)
        except Exception:
            return None
        return BeautifulSoup(html.read(), 'html.parser')
    
    def safeGet(bs,selector):
        selecteds_items = bs.select(selector)
        if selecteds_items is not None and len(selecteds_items) > 0:
            return '\n'.join([elem.get_text() for elem in selecteds_items]) 
        return ''

    def getContent(website,path):
        url = website.url+path
        bs = Crawler.getPage(url)
        if bs is not None:
            title = Crawler.safeGet(bs,website.titleTag)
            body = Crawler.safeGet(bs,website.bodyTag)
            return Content(title,body,url)
        return Content('test', 'test', url)

siteData = [
    Website('Wikipedia', 'https://en.wikipedia.org', 'h1#firstHeading',  'main#content'),
    Website('BBC',       'https://bbc.com',       'h1.story',  'div.body'),
    Website('Reuters',   'https://reuters.com',   'h1.title',  'div.article'),
]

print(siteData[0].url)

# Crawler to Wikipedia webiste and scrape this page `/library/view/.....`
Crawler.getContent(siteData[0],'/wiki/Python_(programming_language)').print()

    

https://en.wikipedia.org
TITLE: Python (programming language)
BODY: 





Toggle the table of contents







Python (programming language)



117 languages




AfrikaansAlemannischAragonésالعربيةঅসমীয়াAsturianuAzərbaycancaتۆرکجهBasa BaliБеларуская (тарашкевіца)БеларускаяБългарскиभोजपुरीবাংলাBrezhonegBosanskiBasa UgiCatalàCebuanoکوردیČeštinaCymraegDanskDeutschKadazandusunΕλληνικάEsperantoEspañolEestiEuskaraفارسیSuomiNa Vosa VakavitiFrançaisGalegoگیلکیગુજરાતીHausaעבריתहिन्दीHrvatskiMagyarՀայերենInterlinguaBahasa IndonesiaIdoÍslenskaItaliano日本語La .lojban.ქართულიQaraqalpaqshaҚазақшаភាសាខ្មែរ한국어KurdîКыргызчаLatinaLombardລາວLietuviųLatviešuМакедонскиമലയാളംМонголमराठीBahasa Melayuမြန်မာဘာသာPlattdüütschनेपालीNederlandsNorsk nynorskNorsk bokmålߒߞߏଓଡ଼ିଆਪੰਜਾਬੀPolskiPiemontèisپنجابیPortuguêsRuna SimiRomânăРусскийСаха тылаᱥᱟᱱᱛᱟᱲᱤScotsSrpskohrvatski / српскохрватскиတႆးසිංහලSimple EnglishSlovenčinaSlovenščinaShqipСрпски / srpskiSvenskaKiswahiliதமிழ்తెలుగుТоҷикӣไทยTagalogToki ponaTürkçeТатарча / tat

In [ ]:
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from bs4 import BeautifulSoup


class Content:
    def __init__(self,topic,title,body,url):
        self.topic = topic
        self.title = title
        self.body = body
        self.url = url
    
    def print(self):
        print(f'topic: {self.topic}')
        print(f'TITLE: {self.title}')
        print(f'BODY: {self.body}')
        print(f'URL: {self.url}')

class Website:
    def __init__(self, name, url, titleTag, bodyTag, absoluteUrl, resultListing, resultTitleTag, resultUrl, searchUrl):
        self.name = name
        self.url = url
        self.titleTag = titleTag        # title tag of the ARTICLE page
        self.bodyTag = bodyTag          # body tag of the ARTICLE page
        self.absoluteUrl = absoluteUrl
        self.resultListing = resultListing      # container of each search result
        self.resultTitleTag = resultTitleTag    # title tag inside each result
        self.resultUrl = resultUrl              # <a> tag inside each result
        self.searchUrl = searchUrl

class Crawler:
    def __init__(self,website):
        self.site = website
        self.found = {}
    
    def getPage(url):
        req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            html = urlopen(req)
        except Exception:
            return None
        return BeautifulSoup(html.read(), 'html.parser')
    
    def safeGet(bs,selector):
        selecteds_items = bs.select(selector)
        if selecteds_items is not None and len(selecteds_items) > 0:
            return '\n'.join([elem.get_text() for elem in selecteds_items])

    def getContent(self,topic,url):
        bs = Crawler.getPage(url)
        if bs is not None:
            title = Crawler.safeGet(bs,self.site.titleTag)
            body = Crawler.safeGet(bs,self.site.bodyTag)
            return Content(topic,title,body,url)
        return Content(topic,'test', 'test', url)

    def search(self,topic):
        '''
        Searches a given website for a given topic and records all pages found
        '''
        bs = Crawler.getPage(self.site.searchUrl + topic) # > Example: `"https://www.reuters.com/site-search/?query="` + `"python"` → fetches the Reuters search results page for "python".
        if bs is None:
            return 'No'
        
        searchResults = bs.select(self.site.resultListing) # Uses the CSS selector stored in `resultListing` to find **all individual result cards/blocks** on the search page. Returns a list of HTML elements.
        for res in searchResults:
            url = res.select(self.site.resultUrl)[0].attrs['href'] # Within the current result block, finds the **link element** using the `resultUrl` CSS selector, takes the first match `[0]`, and extracts its `href` attribute — the raw URL string.
            # check if the URL is relative or an absolute URL
            url = url if self.site.absoluteUrl else self.site.url + url
            if url not in self.found:
                self.found[url] = self.getContent(topic, url)
                self.found[url].print()



# NOTE: Before writing your scraper, you manually inspect the page's HTML (using browser DevTools or viewing page source), check what the href looks like, and then set absoluteUrl accordingly
'''
absoluteUrl = True   →  "Yes, the URLs are absolute"  → do nothing
absoluteUrl = False  →  "No, the URLs are NOT absolute" → fix them



siteData = [
    Website('Reuters', 'https://www.reuters.com/',
            
            'h1' # where the title of the search is for example `Search results for “python”`
            
            ,'span.text-module__text__0GDob text-module__dark-grey__UFC18 text-module__medium__2Rl30 text-module__heading_6__u1KdJ count' # where the number of resulta is stored for example `34 results`
            
            ,False, # Boolen True if the href URLs of the searchs resultas are absolute or False the URLs are NOT absolute
            
            'ul.search-results-module__list__HKgHe li', # where the result of the search is stored and we added 'li' to target individual results

            'div.title-module__title__Lr6MH story-card-module__area-headline__N6EZa', # where the title of the links are stored
            
            'a.text-module__text__0GDob text-module__dark-grey__UFC18 text-module__inherit-font__1P1hv text-module__inherit-size__EyiQW link-module__link__INqxZ link-module__underline_on_hover__YTwYC' # where the href url of the link is stored
            
            ,'https://www.reuters.com/site-search/?query='),
]
'''
# NOTE : scrape only the first page of the resulats not all of them for example if the Pagination Info is `1 to 20 of 34` we scrape only the 1st one not all 20 of them

siteData = [
    Website(
        'Wikipedia',
        'https://en.wikipedia.org',
        'h1',                           # article title
        'div#mw-content-text',          # article body
        False,                          # hrefs are relative
        'ul.mw-search-results li',      # each search result
        'div.mw-search-result-heading', # title inside result
        'a',                            # href link
        'https://en.wikipedia.org/w/index.php?search='
    ),
]

crawlers = [Crawler(site) for site in siteData]
topics = ['data%20science']

for topic in topics:
    for crawler in crawlers:
        crawler.search(topic)


